# Random Forest Classification

## Definition
Random Forest is an **ensemble** method that builds a collection of Decision Trees on **bootstrapped subsets** of training data with **random feature selection** at each split. The final prediction is the **majority vote** across all trees.

## Core Concept

### Two Sources of Randomness
| Source | Description |
|--------|-------------|
| **Bootstrap Sampling** | Each tree is trained on a random sample *with replacement* |
| **Feature Randomness** | At each split, only `√n_features` random features are considered |

### Ensemble Voting
$$\hat{y} = \text{mode}\{h_1(x),\; h_2(x),\; \ldots,\; h_T(x)\}$$

where $h_t$ is the $t$-th tree.

### Why It Works — Bias–Variance Trade-off
- Individual deep trees: **low bias, high variance**
- Averaging many trees: **variance reduces** while bias stays low → better generalisation

### Key Hyper-parameters
| Parameter | Effect |
|-----------|--------|
| `n_estimators` | Number of trees — more = better (diminishing returns) |
| `max_depth` | Depth of each tree |
| `max_features` | Features considered per split (`sqrt`, `log2`) |
| `min_samples_leaf` | Controls smoothness |

## Applications
- **Healthcare** — Disease classification, drug discovery
- **Finance** — Credit scoring, fraud detection
- **Ecology** — Species classification
- **Remote sensing** — Land-use classification from satellite data

---

## Why Iris Dataset?
> `load_iris()` is a clean, balanced 3-class dataset with interpretable features. It lets us directly compare a single Decision Tree vs Random Forest — demonstrating how ensemble averaging reduces variance and improves accuracy. Feature importances from the forest are also highly interpretable on this dataset.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
data = load_iris()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['species'] = df['target'].map(dict(enumerate(data.target_names)))
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
X = df[data.feature_names]; y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Single Decision Tree (baseline)
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print(f'Decision Tree Accuracy : {accuracy_score(y_test, dt.predict(X_test)):.4f}')
print(f'Random Forest Accuracy : {accuracy_score(y_test, rf.predict(X_test)):.4f}')

In [ ]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(cmap='Greens')
plt.title('Random Forest — Confusion Matrix'); plt.tight_layout(); plt.show()

# Feature Importances
imp = pd.Series(rf.feature_importances_, index=data.feature_names).sort_values(ascending=True)
imp.plot(kind='barh', color='forestgreen', edgecolor='k')
plt.title('Random Forest — Feature Importances')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()

# OOB / n_estimators vs accuracy
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf_oob.fit(X_train, y_train)
print(f'Out-of-Bag Score: {rf_oob.oob_score_:.4f}')